In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files
import os
import torch

!pip install -q transformers accelerate bitsandbytes

from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

CKPT_DIR = "/content/drive/MyDrive/llm_results"
os.makedirs(CKPT_DIR, exist_ok=True)

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

Mounted at /content/drive
Using device: cuda


In [ ]:
# ===== DROP-IN REPLACEMENT for the BASE_URL / read_csv cell =====
# Produces train_raw, val_raw, test_raw with columns "Data" and "Label" (0=neutral, 1=positive, 2=negative),
# with the column names the rest of the notebook expects.
!pip install -q datasets
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 17
TOTAL = 15700          # subsample size (about 15.7k rows); set None to use everything

ds = load_dataset("SayedShaun/sentigold")
raw = ds["train"].to_pandas()
print("columns:", raw.columns.tolist())
print(raw["Polarity"].value_counts())

# 5 -> 3 classes, same ids as the notebook's LABEL_MAP (neutral=0, positive=1, negative=2)
COLLAPSE = {"SP": 1, "WP": 1, "NU": 0, "WN": 2, "SN": 2}
unknown = set(raw["Polarity"].unique()) - set(COLLAPSE)
assert not unknown, f"Unexpected Polarity values {unknown}: edit COLLAPSE using the counts printed above"

df = pd.DataFrame({"Data": raw["Text"], "Label": raw["Polarity"].map(COLLAPSE)})
if "Domain" in raw.columns:
    df["Domain"] = raw["Domain"]          # extra column, ignored by clean(); handy for later analysis

# pre-apply the notebook's own filters (non-null, 1 < length < 100, no duplicates) so clean() doesn't shrink it much
df = df.dropna(subset=["Data", "Label"])
df["Data"] = df["Data"].astype(str).str.strip()
df = df[(df["Data"].str.len() > 1) & (df["Data"].str.len() < 100)]
df = df.drop_duplicates(subset="Data").reset_index(drop=True)
df["Label"] = df["Label"].astype(int)
print("after filtering:", len(df), dict(df["Label"].value_counts()))

if TOTAL and len(df) > TOTAL:
    df = df.groupby("Label", group_keys=False).apply(
        lambda d: d.sample(frac=TOTAL / len(df), random_state=SEED)).reset_index(drop=True)

# stratified 80 / 10 / 10
train_raw, rest = train_test_split(df, test_size=0.20, stratify=df["Label"], random_state=SEED)
val_raw, test_raw = train_test_split(rest, test_size=0.50, stratify=rest["Label"], random_state=SEED)
train_raw, val_raw, test_raw = [d.reset_index(drop=True) for d in (train_raw, val_raw, test_raw)]

print(train_raw.shape, val_raw.shape, test_raw.shape)
print("label mix (train/val/test):",
      [dict(d["Label"].value_counts().sort_index()) for d in (train_raw, val_raw, test_raw)])

README.md:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.42MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

columns: ['ID', 'Text', 'Polarity', 'Domain']
Polarity
SP    15318
NU    15156
SN    13767
WN    13682
WP    12077
Name: count, dtype: int64
after filtering: 65257 {2: np.int64(25555), 1: np.int64(25298), 0: np.int64(14404)}
(12559, 3) (1570, 3) (1570, 3)
label mix (train/val/test): [{0: np.int64(2772), 1: np.int64(4869), 2: np.int64(4918)}, {0: np.int64(347), 1: np.int64(608), 2: np.int64(615)}, {0: np.int64(346), 1: np.int64(609), 2: np.int64(615)}]


/tmp/ipykernel_1827/874000056.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("Label", group_keys=False).apply(


In [ ]:
def clean(df):
    df = df.rename(columns={"Data": "sentence", "Label": "sentiment"}).copy()
    df = df.dropna(subset=["sentence", "sentiment"])
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df = df[(df["sentence"].str.len() > 1) & (df["sentence"].str.len() < 100)]
    df = df.drop_duplicates(subset="sentence").reset_index(drop=True)
    return df[["sentence", "sentiment"]]

pool_source = clean(pd.concat([train_raw, val_raw], ignore_index=True))
test_df     = clean(test_raw)

pool_source = pool_source[~pool_source["sentence"].isin(set(test_df["sentence"]))].reset_index(drop=True)

print("Pool source:", len(pool_source))
print("Locked test:", len(test_df))

Pool source: 14129
Locked test: 1570


In [ ]:
def make_split(pool_source, seed_size=200, seed=17):
    frac = seed_size / len(pool_source)
    seed_train = pool_source.groupby("sentiment", group_keys=False).sample(frac=frac, random_state=seed)
    pool_df = pool_source.drop(seed_train.index).reset_index(drop=True)
    return seed_train.reset_index(drop=True), pool_df

seed_train, pool_df = make_split(pool_source, seed_size=200, seed=17)
print("Seed train:", len(seed_train), dict(seed_train["sentiment"].value_counts()))
print("Pool:", len(pool_df))

Seed train: 200 {1: np.int64(78), 2: np.int64(78), 0: np.int64(44)}
Pool: 13929


In [ ]:
def least_confidence(probs):
    p_max = np.max(probs)
    n = len(probs)
    return -(1 - p_max) * (n / (n - 1))

def margin_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] - top2[1]

def ratio_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] / (top2[1] + 1e-12)

def entropy_sampling(probs):
    p = np.clip(probs, 1e-12, 1)
    ent = -np.sum(p * np.log2(p)) / np.log2(len(probs))
    return -ent

def random_sampling(probs, rng):
    return rng.random()

STRATEGIES = {
    "least_confidence": least_confidence,
    "margin": margin_sampling,
    "ratio": ratio_sampling,
    "entropy": entropy_sampling,
    "random": random_sampling,
}

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

llm_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/qwen2.5-7b-instruct")
llm_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/qwen2.5-7b-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
LABEL_MAP = {"neutral": 0, "positive": 1, "negative": 2}
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

def build_prompt(sentence, examples=None):
    instructions = (
        "You are a sentiment classifier for Bangla text. "
        "Classify the sentiment of the given text as exactly one of: positive, negative, neutral. "
        "Respond with only the single label word, nothing else."
    )
    few_shot_block = ""
    if examples:
        for ex_text, ex_label in examples:
            few_shot_block += f"\nText: {ex_text}\nLabel: {LABEL_NAMES[ex_label]}"
    return f"{instructions}\n{few_shot_block}\n\nText: {sentence}\nLabel:"


In [ ]:
def make_chat_text(sentence, examples=None):
    msgs = [{"role": "user", "content": build_prompt(sentence, examples)}]
    return llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

llm_tokenizer.padding_side = "left"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

LABEL_WORDS = [LABEL_NAMES[i] for i in range(3)]
LABEL_TOKEN_IDS = [llm_tokenizer.encode(w, add_special_tokens=False)[0] for w in LABEL_WORDS]
assert len(set(LABEL_TOKEN_IDS)) == 3, "label first-tokens collide; use a different label word"

In [ ]:
def atomic_pickle(obj, path):
    """Write to a temp file, then swap in. A disconnect mid-write can't corrupt the real checkpoint."""
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

In [ ]:
@torch.no_grad()
def llm_annotate_rows(rows_df, examples=None, batch_size=16):
    """Returns (labels, confidences). Replaces the older version that returned labels only."""
    texts = [make_chat_text(s, examples) for s in rows_df["sentence"].tolist()]
    out = np.zeros((len(texts), 3), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        enc = llm_tokenizer(texts[i:i+batch_size], return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]
        out[i:i+batch_size] = torch.softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
    return out.argmax(axis=1), out.max(axis=1)

In [ ]:
MODEL_NAME = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_texts(texts, max_length=64):
    return tokenizer(list(texts), truncation=True, padding=True, max_length=max_length, return_tensors="pt")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# def make_bert_model():
#     model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
#     return model.to(device)

from transformers import set_seed

def make_bert_model(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    return model.to(device)

In [ ]:
def fine_tune_round(train_df, seed, epochs=3, batch_size=16):
    model = make_bert_model(seed)

    train_encodings = tokenize_texts(train_df["sentence"])
    train_dataset = Dataset.from_dict({
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"],
        "labels": train_df["sentiment"].tolist(),
    })

    # args = TrainingArguments(
    #     output_dir="/content/bert_tmp",
    #     num_train_epochs=epochs,
    #     per_device_train_batch_size=batch_size,
    #     logging_steps=50,
    #     save_strategy="no",
    #     report_to="none",
    #     seed=seed,
    # )

    args = TrainingArguments(
        output_dir="/content/bert_tmp",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=2e-5,      # down from the Trainer default of 5e-5
        # warmup_ratio=0.1,        # ease into training instead of a hard start
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
    trainer.train()
    return model

In [ ]:
import torch.nn.functional as F

def predict_probs(model, sentences, batch_size=32):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            enc = tokenize_texts(batch)
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

In [ ]:
# ===== STEP 5, GATE 1: AL with the frozen few-shot + calibration annotator (+ class-balanced random control) =====
# Needs (already in your notebook): seed_train, pool_df, test_df, STRATEGIES, fine_tune_round, predict_probs,
#   llm_model, llm_tokenizer, make_chat_text, LABEL_TOKEN_IDS, LABEL_NAMES, CKPT_DIR, atomic_pickle
# Needs on Drive: step2_logprobs.npz, step3_fs_k4_s17.npz, sentigold_human_annotator_margin_annotations.csv

# ---- CELL 5.0: freeze the annotator ONCE (12 examples + 2 offsets) ----
import os, json, time, hashlib, pickle
import numpy as np, pandas as pd, torch
from scipy.optimize import minimize
from scipy.special import log_softmax, softmax
from sklearn.metrics import f1_score, accuracy_score

ANN_PATH = f"{CKPT_DIR}/step5_annotator.json"
if os.path.exists(ANN_PATH):
    ann = json.load(open(ANN_PATH)); print("annotator was already frozen -> loaded it, not changed")
else:
    z = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)
    f = np.load(f"{CKPT_DIR}/step3_fs_k4_s17.npz", allow_pickle=True)
    keep, ys = f["keep_seed"], z["y_seed"].astype(int)
    assert (seed_train["sentiment"].values == ys).all(), "seed_train order differs from the Step 2 file"
    def pick_shots(k_per_class=4, seed=17):                           # identical to Step 3 / your 3a notebook
        shots = []
        for lab in range(3):
            rows = seed_train[seed_train.sentiment == lab].sample(k_per_class, random_state=seed)
            shots += list(zip(rows.sentence, rows.sentiment))
        rng = np.random.default_rng(seed); rng.shuffle(shots)
        return shots
    shots = pick_shots(4, 17)
    assert {s for s, _ in shots} == set(seed_train["sentence"][~keep]), "examples differ from the Step 3 draw 17"
    def shift(LP, b): return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)
    def fit(LP, y):
        ce = lambda b: -shift(LP, b)[np.arange(len(y)), y].mean()
        return minimize(ce, x0=np.zeros(2), method="Nelder-Mead").x
    b = fit(f["LP_seed"][keep], ys[keep])                            # few-shot scores of the 188 seed rows NOT in the prompt
    ann = {"shots": [[s, int(l)] for s, l in shots], "offsets": {"neutral": float(b[0]), "negative": float(b[1])},
           "draw_seed": 17, "k_per_class": 4}
    json.dump(ann, open(ANN_PATH, "w"), ensure_ascii=False)
    print("annotator frozen.")
print("offsets:", {k: round(v, 2) for k, v in ann["offsets"].items()}, "  (Step 3 printed about [-3.1, -2.94] for draw 17)")
print("examples:", [LABEL_NAMES[l] for _, l in ann["shots"]])

# class mix for the balanced-random control = class shares of the rows the HUMAN margin run queried
a = pd.read_csv(f"{CKPT_DIR}/sentigold_human_annotator_margin_annotations.csv")
CLASS_MIX = a["true_label"].value_counts(normalize=True).sort_index().values
print("class mix of human-margin queries (neutral/positive/negative):", np.round(CLASS_MIX, 3))


annotator was already frozen -> loaded it, not changed
offsets: {'neutral': -3.1, 'negative': -2.94}   (Step 3 printed about [-3.1, -2.94] for draw 17)
examples: ['negative', 'positive', 'positive', 'negative', 'neutral', 'neutral', 'negative', 'positive', 'negative', 'neutral', 'positive', 'neutral']
class mix of human-margin queries (neutral/positive/negative): [0.361 0.349 0.289]


In [ ]:
@torch.no_grad()
def llm_annotate_rows_fs(rows_df, ann, batch_size=8):
    """Frozen annotator: the 12 fixed examples in the prompt, then the two calibration offsets, then argmax."""
    llm_tokenizer.padding_side = "left"
    shots = [(t, int(l)) for t, l in ann["shots"]]
    off = np.array([ann["offsets"]["neutral"], 0.0, ann["offsets"]["negative"]], dtype=np.float32)
    texts = [make_chat_text(s, shots) for s in rows_df["sentence"].tolist()]
    out = np.zeros((len(texts), 3), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        enc = llm_tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]
        lp = torch.log_softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy() + off
        out[i:i + batch_size] = softmax(lp, axis=1)
    return out.argmax(axis=1), out.max(axis=1)

def run_bert_active_learning_v2(seed_train, pool_df, test_df, strategy_name, annotator="llm_fs",
                                n_rounds=15, batch_size=100, seed=17, run_id=None, ann=None, class_mix=None):
    assert annotator in ("llm", "human", "llm_fs")
    if strategy_name == "balanced_random":
        # assert annotator == "human" and class_mix is not None, "balanced_random needs gold labels (human) and a class mix"
        assert annotator in ("human", "llm_fs") and class_mix is not None, "balanced_random needs a class mix"
    if annotator == "llm_fs":
        assert ann is not None
    run_id = run_id or f"sentigold_{annotator}_annotator_{strategy_name}"
    ann_id = hashlib.sha1(json.dumps(ann, sort_keys=True).encode()).hexdigest()[:12] if annotator == "llm_fs" else "none"
    state_path = f"{CKPT_DIR}/{run_id}.pkl"
    strategy_fn = STRATEGIES.get(strategy_name)
    rng = np.random.default_rng(seed)

    train_df = pool = logs = agreement_log = None
    ann_df, start_round = pd.DataFrame(), 0
    if os.path.exists(state_path):
        try:
            with open(state_path, "rb") as fh:
                st = pickle.load(fh)
            assert st.get("annotator_id", "none") == ann_id, "this checkpoint was made with a DIFFERENT annotator"
            train_df, pool = st["train_df"], st["pool"]
            logs, agreement_log = st["logs"], st["agreement_log"]
            ann_df = st.get("annotations", pd.DataFrame())
            if st.get("rng_state") is not None:
                rng.bit_generator.state = st["rng_state"]
            start_round = st["round"] + 1
            print(f"Resuming {run_id} from round {start_round}")
        except AssertionError:
            raise                                                    # never silently mix annotators
        except Exception as e:
            print(f"Checkpoint unreadable ({e!r}) -- starting {run_id} fresh.")
            train_df = None
    if train_df is None:
        train_df, pool = seed_train.copy().reset_index(drop=True), pool_df.copy().reset_index(drop=True)
        logs, agreement_log, ann_df, start_round = [], [], pd.DataFrame(), 0

    for round_i in range(start_round, n_rounds):
        if len(pool) < batch_size:
            break
        t0 = time.time()
        model = fine_tune_round(train_df, seed=seed, epochs=3, batch_size=16)
        test_preds = np.argmax(predict_probs(model, test_df["sentence"].tolist()), axis=1)
        f1m = f1_score(test_df["sentiment"].values, test_preds, average="macro")
        acc = accuracy_score(test_df["sentiment"].values, test_preds)
        logs.append({"strategy": strategy_name, "annotator": annotator, "seed": seed, "round": round_i,
                     "train_size": len(train_df), "test_accuracy": acc, "test_f1_macro": f1m})
        print(f"[{run_id}] round {round_i} | train_size={len(train_df)} | test_f1_macro={f1m:.3f} acc={acc:.3f}")

        if strategy_name == "balanced_random":                       # random rows, but with a fixed gold class mix
            counts = np.round(np.asarray(class_mix) * batch_size).astype(int)
            counts[np.argmax(counts)] += batch_size - counts.sum()
            gold_pool = pool["sentiment"].values
            query_idx = np.concatenate([rng.choice(np.where(gold_pool == c)[0], size=counts[c], replace=False) for c in range(3)])
        else:
            pool_probs = predict_probs(model, pool["sentence"].tolist())
            if strategy_name == "random":
                scores = np.array([strategy_fn(p, rng) for p in pool_probs])
            else:
                scores = np.array([strategy_fn(p) for p in pool_probs])
            query_idx = np.argsort(scores)[:batch_size]
        queried = pool.iloc[query_idx].copy()

        true_labels = queried["sentiment"].values.copy()
        if annotator == "llm_fs":
            train_labels, conf = llm_annotate_rows_fs(queried, ann)
        elif annotator == "llm":
            train_labels, conf = llm_annotate_rows(queried, examples=None)
        else:
            train_labels, conf = true_labels.copy(), np.ones(len(queried), dtype=np.float32)
        agreement = float((true_labels == train_labels).mean())
        agreement_log.append({"round": round_i, "n_labeled": len(queried), "llm_agreement": agreement})
        print(f"    annotator={annotator} agreement with gold this round: {agreement:.3f}")

        ann_df = pd.concat([ann_df, pd.DataFrame({
            "round": round_i, "sentence": queried["sentence"].values,
            "true_label": true_labels, "train_label": train_labels, "annotator_conf": conf})], ignore_index=True)

        queried["sentiment"] = train_labels
        pool = pool.drop(pool.index[query_idx]).reset_index(drop=True)
        train_df = pd.concat([train_df, queried], ignore_index=True)

        logs[-1]["round_seconds"] = time.time() - t0
        atomic_pickle({"round": round_i, "train_df": train_df, "pool": pool, "logs": logs,
                       "agreement_log": agreement_log, "annotations": ann_df,
                       "rng_state": rng.bit_generator.state, "annotator_id": ann_id}, state_path)
        print(f"    round took {logs[-1]['round_seconds'] / 60:.1f} min")
        del model
        torch.cuda.empty_cache()

    curve = pd.DataFrame(logs)
    curve.to_csv(f"{CKPT_DIR}/{run_id}_curve.csv", index=False)
    ann_df.to_csv(f"{CKPT_DIR}/{run_id}_annotations.csv", index=False)
    return curve, pd.DataFrame(agreement_log), ann_df

In [ ]:
RUNS5 = [("llm_fs", "margin"), ("llm_fs", "entropy"), ("llm_fs", "random"), ("human", "balanced_random")]
results5 = {}
for annotator, strat in RUNS5:
    rid = f"sentigold_{annotator}_annotator_{strat}"
    t_run = time.time()
    results5[rid] = run_bert_active_learning_v2(seed_train, pool_df, test_df, strategy_name=strat, annotator=annotator,
                                                n_rounds=15, batch_size=100, seed=17, run_id=rid,
                                                ann=ann if annotator == "llm_fs" else None,
                                                class_mix=CLASS_MIX if strat == "balanced_random" else None)
    print(f"### {rid} done; this session spent {(time.time() - t_run) / 3600:.2f} h on it (~{(time.time() - t_run) / 3600 * 1.08:.1f} units)")

Resuming sentigold_llm_fs_annotator_margin from round 15
### sentigold_llm_fs_annotator_margin done; this session spent 0.00 h on it (~0.0 units)
Resuming sentigold_llm_fs_annotator_entropy from round 1


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

... (171 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,1.090574
100,0.995678
150,0.769418
200,0.667368
250,0.539256
300,0.474899


[sentigold_human_annotator_balanced_random] round 14 | train_size=1600 | test_f1_macro=0.728 acc=0.741
    annotator=human agreement with gold this round: 1.000
    round took 0.6 min
### sentigold_human_annotator_balanced_random done; this session spent 0.08 h on it (~0.1 units)


In [ ]:
import numpy as np, pandas as pd, os

NAMES = [("human", "random"), ("human", "margin"), ("human", "entropy"), ("human", "balanced_random"),
         ("llm", "random"), ("llm", "margin"), ("llm", "entropy"),
         ("llm_fs", "random"), ("llm_fs", "margin"), ("llm_fs", "entropy")]
rows, curves = [], {}
for ann_, strat in NAMES:
    rid = f"sentigold_{ann_}_annotator_{strat}"
    cp, ap = f"{CKPT_DIR}/{rid}_curve.csv", f"{CKPT_DIR}/{rid}_annotations.csv"
    if not (os.path.exists(cp) and os.path.exists(ap)):
        print(f"missing: {rid}"); continue
    c, a = pd.read_csv(cp), pd.read_csv(ap)
    curves[(ann_, strat)] = c
    late = c[c["round"] >= 10]
    secs = c["round_seconds"].sum() / 3600 if "round_seconds" in c else np.nan
    rows.append({"annotator": ann_, "strategy": strat, "rounds": len(c),
                 "F1 r10-14": late["test_f1_macro"].mean(), "acc r10-14": late["test_accuracy"].mean(),
                 "F1 last": c["test_f1_macro"].iloc[-1],
                 "label agreement": (a["true_label"] == a["train_label"]).mean(),
                 "gold neutral share of queries": (a["true_label"] == 0).mean(),
                 "label neutral share": (a["train_label"] == 0).mean(),
                 "hours": secs})
T = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(T.round(3).to_string(index=False))

print("\nGain over RANDOM with the same annotator (mean F1 of rounds 10-14, in F1 points):")
for ann_ in ("human", "llm", "llm_fs"):
    r = T[(T.annotator == ann_) & (T.strategy == "random")]
    if r.empty: continue
    base = r["F1 r10-14"].iloc[0]
    for strat in ("margin", "entropy"):
        m = T[(T.annotator == ann_) & (T.strategy == strat)]
        if not m.empty: print(f"  {ann_:7s} {strat:8s} {100 * (m['F1 r10-14'].iloc[0] - base):+.1f}")
b = T[(T.annotator == "human") & (T.strategy == "balanced_random")]
hr = T[(T.annotator == "human") & (T.strategy == "random")]
if not b.empty and not hr.empty:
    print(f"\nclass-balanced random (human labels) vs plain random (human labels): {100 * (b['F1 r10-14'].iloc[0] - hr['F1 r10-14'].iloc[0]):+.1f} F1 points")
    for strat in ("margin", "entropy"):
        m = T[(T.annotator == "human") & (T.strategy == strat)]
        if not m.empty: print(f"human {strat} vs class-balanced random: {100 * (m['F1 r10-14'].iloc[0] - b['F1 r10-14'].iloc[0]):+.1f} F1 points")
print(f"\ntotal GPU hours recorded by the new runs: {T['hours'].sum():.2f}  (~{T['hours'].sum() * 1.08:.1f} units at 1.08/h)")

annotator        strategy  rounds  F1 r10-14  acc r10-14  F1 last  label agreement  gold neutral share of queries  label neutral share  hours
    human          random      15      0.696       0.742    0.722            1.000                          0.234                0.234    NaN
    human          margin      15      0.736       0.764    0.748            1.000                          0.361                0.361    NaN
    human         entropy      15      0.729       0.755    0.738            1.000                          0.349                0.349    NaN
    human balanced_random      15      0.723       0.739    0.728            1.000                          0.360                0.360  0.080
      llm          random      15      0.691       0.704    0.687            0.695                          0.234                0.388    NaN
      llm          margin      15      0.693       0.703    0.694            0.590                          0.238                0.447    NaN
      

In [ ]:
# ===== SEED 18, CELL S0: new seed set + its own frozen annotator (needs the notebook state from cells 5.0 and 5.1) =====
import os, json, time
import numpy as np, pandas as pd, torch
from scipy.optimize import minimize
from scipy.special import log_softmax

S = 18
seed_train_s, pool_df_s = make_split(pool_source, seed_size=200, seed=S)       # same function as for seed 17, different seed
print("seed set:", len(seed_train_s), dict(seed_train_s["sentiment"].value_counts().sort_index()), "| pool:", len(pool_df_s))
assert not set(seed_train_s["sentence"]) & set(test_df["sentence"]), "seed rows leak into the test set"
assert not set(pool_df_s["sentence"]) & set(seed_train_s["sentence"]), "pool and seed overlap"
print("rows shared with the seed-17 seed set (just for information):", len(set(seed_train_s["sentence"]) & set(seed_train["sentence"])))

def freeze_annotator_for_seed(S, seed_df, k_per_class=4):
    """12 fixed examples drawn from THIS seed set + offsets fitted on the 188 seed rows not shown in the prompt (same recipe as seed 17)."""
    path = f"{CKPT_DIR}/step5_annotator_s{S}.json"
    if os.path.exists(path):
        print(f"seed {S}: annotator already frozen -> loaded, not changed"); return json.load(open(path))
    shots = []
    for lab in range(3):
        rows = seed_df[seed_df.sentiment == lab].sample(k_per_class, random_state=S)
        shots += list(zip(rows.sentence, rows.sentiment))
    np.random.default_rng(S).shuffle(shots)
    shot_sents = {s for s, _ in shots}
    held = seed_df[~seed_df["sentence"].isin(shot_sents)].reset_index(drop=True)
    llm_tokenizer.padding_side = "left"
    texts = [make_chat_text(s, [(t, int(l)) for t, l in shots]) for s in held["sentence"]]

    @torch.no_grad()
    def score(batch_texts):
        enc = llm_tokenizer(batch_texts, return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]
        return torch.log_softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
    LP = np.concatenate([score(texts[i:i + 8]) for i in range(0, len(texts), 8)])
    y = held["sentiment"].values
    shift = lambda LPx, b: log_softmax(LPx + np.array([b[0], 0.0, b[1]]), axis=1)
    ce = lambda b: -shift(LP, b)[np.arange(len(y)), y].mean()
    b = minimize(ce, x0=np.zeros(2), method="Nelder-Mead").x
    ann_new = {"shots": [[s, int(l)] for s, l in shots], "offsets": {"neutral": float(b[0]), "negative": float(b[1])},
               "draw_seed": S, "k_per_class": k_per_class}
    json.dump(ann_new, open(path, "w"), ensure_ascii=False)
    print(f"seed {S}: annotator frozen. raw few-shot accuracy on the {len(y)} held-out seed rows: {np.mean(LP.argmax(1) == y):.3f}")
    return ann_new

ann_s = freeze_annotator_for_seed(S, seed_train_s)
print("seed", S, "offsets:", {k: round(v, 2) for k, v in ann_s["offsets"].items()}, "(seed 17 had -3.1 / -2.94)")
print("examples:", [LABEL_NAMES[l] for _, l in ann_s["shots"]])

seed set: 200 {0: np.int64(44), 1: np.int64(78), 2: np.int64(78)} | pool: 13929
rows shared with the seed-17 seed set (just for information): 4
seed 18: annotator already frozen -> loaded, not changed
seed 18 offsets: {'neutral': -2.33, 'negative': -2.34} (seed 17 had -3.1 / -2.94)
examples: ['negative', 'positive', 'neutral', 'neutral', 'positive', 'positive', 'positive', 'negative', 'neutral', 'negative', 'negative', 'neutral']


In [ ]:
# ===== SEED 18, CELL S1: all remaining runs in priority order. Safe to re-run: finished runs are skipped, unfinished ones resume. =====
import time, os
import numpy as np, pandas as pd

N_ROUNDS = 15
MAX_UNITS = 20          # stop STARTING new runs once this cell has used about this many units (a run in progress finishes)
t_start = time.time()
units_used = lambda: (time.time() - t_start) / 3600 * 1.08

CTX = {17: dict(seed_df=seed_train, pool=pool_df, ann=ann, prefix="sentigold", mix=lambda: CLASS_MIX),
       18: dict(seed_df=seed_train_s, pool=pool_df_s, ann=ann_s, prefix="sentigold_s18",
                mix=lambda: pd.read_csv(f"{CKPT_DIR}/sentigold_s18_human_annotator_margin_annotations.csv")["true_label"]
                                .value_counts(normalize=True).sort_index().values)}
PLAN = [(17, "llm_fs", "balanced_random"),                                    # the run from before, if it is not done yet
        (18, "human", "margin"), (18, "human", "entropy"), (18, "human", "random"), (18, "human", "balanced_random"),
        (18, "llm_fs", "margin"), (18, "llm_fs", "entropy"), (18, "llm_fs", "random"), (18, "llm_fs", "balanced_random"),
        (18, "llm", "margin"), (18, "llm", "entropy"), (18, "llm", "random")]  # last 3 = zero-shot baseline

results_seed = globals().get("results_seed", {})
try:
    for S_, annotator, strat in PLAN:
        c = CTX[S_]
        rid = f"{c['prefix']}_{annotator}_annotator_{strat}"
        curve_path = f"{CKPT_DIR}/{rid}_curve.csv"
        if os.path.exists(curve_path) and len(pd.read_csv(curve_path)) >= N_ROUNDS:
            print(f"skip (already finished): {rid}"); continue
        if units_used() > MAX_UNITS:
            print(f"STOP: this cell has used about {units_used():.1f} units (limit {MAX_UNITS}). Re-run the cell later to continue."); break
        t0 = time.time()
        results_seed[rid] = run_bert_active_learning_v2(
            c["seed_df"], c["pool"], test_df, strategy_name=strat, annotator=annotator, n_rounds=N_ROUNDS, batch_size=100,
            seed=S_, run_id=rid, ann=c["ann"] if annotator == "llm_fs" else None,
            class_mix=c["mix"]() if strat == "balanced_random" else None)
        print(f"### {rid}: {(time.time() - t0) / 3600:.2f} h | this cell so far ~{units_used():.1f} units")
finally:
    # Releases the GPU so units stop being used while you sleep. Drive checkpoints stay.
    # Delete these 3 lines if you do NOT want the notebook to disconnect itself.
    from google.colab import runtime
    runtime.unassign()

skip (already finished): sentigold_llm_fs_annotator_balanced_random
skip (already finished): sentigold_s18_human_annotator_margin
skip (already finished): sentigold_s18_human_annotator_entropy
skip (already finished): sentigold_s18_human_annotator_random
skip (already finished): sentigold_s18_human_annotator_balanced_random
skip (already finished): sentigold_s18_llm_fs_annotator_margin
skip (already finished): sentigold_s18_llm_fs_annotator_entropy
skip (already finished): sentigold_s18_llm_fs_annotator_random
skip (already finished): sentigold_s18_llm_fs_annotator_balanced_random
skip (already finished): sentigold_s18_llm_annotator_margin
skip (already finished): sentigold_s18_llm_annotator_entropy
skip (already finished): sentigold_s18_llm_annotator_random


In [ ]:
import numpy as np, pandas as pd, os

SEEDS = {17: "sentigold", 18: "sentigold_s18"}
KINDS = ("random", "margin", "entropy", "balanced_random")
RUNS = []
for a in ("human", "llm", "llm_fs"):
    for s in KINDS:
        if not (a == "llm" and s == "balanced_random"):
            RUNS.append((a, s))

rec = []
for seed, pre in SEEDS.items():
    for a, s in RUNS:
        rid = f"{pre}_{a}_annotator_{s}"
        cp = f"{CKPT_DIR}/{rid}_curve.csv"
        ap = f"{CKPT_DIR}/{rid}_annotations.csv"
        if not (os.path.exists(cp) and os.path.exists(ap)):
            print(f"missing: {rid}")
            continue
        c = pd.read_csv(cp)
        an = pd.read_csv(ap)
        if len(c) < 15:
            print(f"unfinished ({len(c)} rounds): {rid}")
        late = c[c["round"] >= 10]
        secs = np.nan
        if "round_seconds" in c:
            secs = c["round_seconds"].sum() / 3600
        rec.append(dict(
            seed=seed, annotator=a, strategy=s,
            f1=late["test_f1_macro"].mean(),
            acc=late["test_accuracy"].mean(),
            agree=(an["true_label"] == an["train_label"]).mean(),
            neu_label=(an["train_label"] == 0).mean(),
            neu_gold=(an["true_label"] == 0).mean(),
            hours=secs))
R = pd.DataFrame(rec)
order = pd.MultiIndex.from_tuples(RUNS, names=["annotator", "strategy"])
pd.set_option("display.width", 200)

def pivot(col):
    P = R.pivot_table(index=["annotator", "strategy"],
                      columns="seed", values=col)
    return P.reindex(order).dropna(how="all")

def table(col, title, only=None):
    P = pivot(col)
    if only:
        P = P[P.index.get_level_values(0).isin(only)]
    seeds_only = P.copy()
    P["mean"] = seeds_only.mean(axis=1)
    P["spread"] = seeds_only.max(axis=1) - seeds_only.min(axis=1)
    show = P.apply(lambda col_: col_.map(
        lambda v: "" if pd.isna(v) else f"{v:.3f}"))
    print("\n" + title)
    print(show.to_string())

def line(vals):
    first = "  ".join(f"{v:+.1f}" for v in vals)
    return first + "   | mean " + f"{np.mean(vals):+.1f}"

table("f1", "macro-F1, mean of rounds 10-14 (spread = max - min over seeds)")
table("acc", "accuracy, mean of rounds 10-14")
table("agree", "label agreement with gold on queried rows",
      only=("llm", "llm_fs"))
table("neu_label", "share of labels that are NEUTRAL",
      only=("llm", "llm_fs"))
table("neu_gold", "gold neutral share of the queried rows",
      only=("llm_fs",))

print("\nGAINS in F1 points per seed, then mean (same annotator):")
for a in ("human", "llm", "llm_fs"):
    for base in ("random", "balanced_random"):
        if a == "llm" and base == "balanced_random":
            continue
        for s in ("margin", "entropy"):
            vals = []
            for seed in SEEDS:
                sub = R[(R.seed == seed) & (R.annotator == a)]
                x = sub.set_index("strategy")["f1"]
                if s in x and base in x:
                    vals.append(100 * (x[s] - x[base]))
            if vals:
                head = f"  {a:7s} {s:8s} vs {base:15s}: "
                print(head + line(vals))

print("\nGAP to human labels in F1 points (human minus annotator):")
for a in ("llm", "llm_fs"):
    for s in ("random", "margin", "entropy"):
        vals = []
        for seed in SEEDS:
            r_ = R[(R.seed == seed) & (R.strategy == s)]
            h = r_[r_.annotator == "human"]["f1"]
            x = r_[r_.annotator == a]["f1"]
            if len(h) and len(x):
                vals.append(100 * (h.iloc[0] - x.iloc[0]))
        if vals:
            head = f"  {a:7s} {s:8s}: "
            print(head + line(vals))

H = pivot("hours")
print("\nhours per run (only runs with timing recorded):")
print(H.round(2).to_string())
tot = np.nansum(H.values)
print(f"total recorded: {tot:.1f} h  (~{tot * 1.08:.1f} units)")


macro-F1, mean of rounds 10-14 (spread = max - min over seeds)
seed                          17     18   mean spread
annotator strategy                                   
human     random           0.696  0.599  0.647  0.097
          margin           0.736  0.724  0.730  0.012
          entropy          0.729  0.721  0.725  0.008
          balanced_random  0.723  0.725  0.724  0.002
llm       random           0.691  0.673  0.682  0.018
          margin           0.693  0.673  0.683  0.021
          entropy          0.645  0.636  0.641  0.008
llm_fs    random           0.652  0.643  0.648  0.009
          margin           0.695  0.701  0.698  0.006
          entropy          0.709  0.690  0.699  0.019
          balanced_random  0.699  0.670  0.684  0.029

accuracy, mean of rounds 10-14
seed                          17     18   mean spread
annotator strategy                                   
human     random           0.742  0.702  0.722  0.041
          margin           0.764  0.749 

In [ ]:
import pandas as pd, numpy as np

NAMES = {0: "neutral", 1: "positive", 2: "negative"}
FILES = {"s17 entropy": "sentigold_llm_fs_annotator_entropy",
         "s17 margin":  "sentigold_llm_fs_annotator_margin",
         "s18 entropy": "sentigold_s18_llm_fs_annotator_entropy",
         "s18 margin":  "sentigold_s18_llm_fs_annotator_margin"}

parts = []
for tag, rid in FILES.items():
    a = pd.read_csv(f"{CKPT_DIR}/{rid}_annotations.csv")
    a["file"] = tag
    parts.append(a)
A = pd.concat(parts, ignore_index=True)
A["gold"] = A["true_label"].map(NAMES)
A["qwen"] = A["train_label"].map(NAMES)
D = A[A["true_label"] != A["train_label"]]

print(f"rows: {len(A)} | disagreements: {len(D)} ({len(D) / len(A):.1%})")
print("\nwhat kind of mistakes (gold -> Qwen), share of all disagreements:")
print((D.groupby(["gold", "qwen"]).size() / len(D)).round(3).sort_values(ascending=False).to_string())

print("\n40 sample disagreements (10 per file):")
S = pd.concat([g.sample(min(10, len(g)), random_state=17) for _, g in D.groupby("file")])
for _, r in S.iterrows():
    print(f"[{r['file']}] gold={r['gold']:8s} qwen={r['qwen']:8s} conf={r['annotator_conf']:.2f} | {r['sentence']}")
S[["file", "gold", "qwen", "annotator_conf", "sentence"]].to_csv(f"{CKPT_DIR}/error_sample_40.csv", index=False)